# Air Quality Data Analysis with Python
## Notebook 7 · Mapping a Sensor Network

⏱️ About 70 minutes &nbsp;·&nbsp; ⬅️ Builds on Notebooks 3–5 &nbsp;·&nbsp; 🗺️ Space, not time

Everything so far has been about **time**: one sensor, one line, one daily cycle.
But a city is not one point. The moment you have a *network* of sensors, a new
question opens up: *is the pollution the same everywhere, or does each
neighbourhood have its own?*

That question has real consequences. If concentrations are roughly uniform, one
monitor represents the city and a city-wide policy makes sense. If they vary
sharply from district to district, one monitor represents one district, and the
fix has to be local too.

We're changing cities for this notebook and the next. **Accra, Ghana** has a
denser network than Lagos and — as you'll verify yourself in section 6 — much
healthier instruments, which matters enormously once you start comparing sensors
to each other.

In this notebook you'll build the three maps that answer the question:

* a **station map** — each sensor coloured by its mean concentration,
* an **interpolated surface** — inverse-distance weighting, to fill the gaps
  between sensors (and to see where you shouldn't trust the filling),
* a **nearest-neighbour correlation map** — do neighbouring sensors rise and
  fall *together*?

And you'll build each one twice: for **February 2025**, deep in the Harmattan,
and for **August 2025**, the heart of the wet season.

### 1. Setup

Two new data files, and one new kind of file — a picture.

In [ ]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from matplotlib.colors import LinearSegmentedColormap

DATA = "../data"  # local checkout of the course repository
if not Path(DATA).exists():  # running in Colab -> read from GitHub
    DATA = "https://raw.githubusercontent.com/rwpinder/tutorial-air-quality-data-analysis/main/data"

GRAY, BLUE, ORANGE, GREEN = "#999999", "#0072B2", "#D55E00", "#009E73"
plt.rcParams.update({
    "figure.figsize": (9.5, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#cbcbcb", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": "#333333", "xtick.color": "#333333", "ytick.color": "#333333",
    "legend.frameon": False,
})


def check(name, test, hint=""):
    """Run test() and print a friendly ✅ or 💡 — never an error message."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    if ok:
        print(f"✅ {name} — looks right!")
    else:
        print(f"💡 {name} — not quite yet. Hint: {hint}")


def read_csv(name):
    return pd.read_csv(f"{DATA}/{name}")


feb = read_csv("accra_network_feb2025.csv")
aug = read_csv("accra_network_aug2025.csv")
for df in (feb, aug):
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True)

sites = read_csv("accra_network_sites.csv").set_index("site_name")

print(f"February 2025: {len(feb):>6} rows, {feb['site_name'].nunique()} sites")
print(f"August 2025:   {len(aug):>6} rows, {aug['site_name'].nunique()} sites")
print(f"Site registry: {len(sites)} sites total")
sites.head()

These are **long format** tables, the same shape as `lagos_sites_recent.csv` in
Notebook 5: one row per site per hour, with the site's name in a column.

The site registry carries the two numbers a map needs — `latitude` and
`longitude` — plus how complete each sensor was in each month.

A note on time zones: Accra is on **UTC+0** all year, so for once UTC and local
time are the same. We'll still write `tz_convert("Africa/Accra")` where it
matters, because relying on a coincidence is how you get burned the first time
you reuse the code somewhere else.

#### Which sensors are here

A sensor earns its place by reporting at least **70%** of the month's hours —
the *same* bar in both months. Anything flagged by the four-tier QA screen from
Notebook 3 doesn't count towards that total, so a sensor that reported
constantly but reported nonsense does not qualify.

In [ ]:
print(sites[["completeness_feb2025", "completeness_aug2025",
             "in_feb2025", "in_aug2025"]].to_string())

### 2. A map is just a scatter plot with a picture behind it

You do not need a mapping library to draw a useful map. A map is:

1. an image, drawn to fill a known rectangle of longitude and latitude, and
2. a scatter plot of points, in those same coordinates, on top.

The course ships a pre-rendered picture of Accra (`accra_basemap.png`) and the
rectangle it covers (`accra_basemap.csv`), so there is nothing to install and
nothing to download while you work.

In [ ]:
def load_png(path):
    """Read a PNG from a local path or a URL."""
    if path.startswith("http"):
        return plt.imread(io.BytesIO(requests.get(path, timeout=60).content), format="png")
    return plt.imread(path)


basemap = load_png(f"{DATA}/accra_basemap.png")
bounds = read_csv("accra_basemap.csv").iloc[0]
EXTENT = [bounds.west, bounds.east, bounds.south, bounds.north]

print(f"Basemap image: {basemap.shape[1]} x {basemap.shape[0]} pixels")
print(f"covering longitude {bounds.west:.3f} to {bounds.east:.3f}, "
      f"latitude {bounds.south:.3f} to {bounds.north:.3f}")

fig, ax = plt.subplots(figsize=(6.8, 6.0))
ax.imshow(basemap, extent=EXTENT)
ax.scatter(sites["longitude"], sites["latitude"], s=60, color=ORANGE,
           edgecolor="white", linewidth=1.5, zorder=3)
ax.set_xticks([]); ax.set_yticks([])
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title("The network, from Amasaman inland to the Gulf of Guinea shore")
plt.show()

`extent=` is the whole trick. It tells matplotlib "stretch this image across
*these* longitude and latitude values", after which a point at (-0.20, 5.55)
lands exactly where longitude -0.20 and latitude 5.55 are on the picture.

Two housekeeping details that make maps look like maps:

* `ax.grid(False)` and empty ticks — a map has no need for a numeric grid,
* `zorder=3` — draws the points *above* the image (higher = nearer the viewer).

Below is a small helper that repeats that setup, so the rest of the notebook can
say `map_panel(ax)` and get on with the data. Read it, but don't worry about
memorising it.

In [ ]:
def map_panel(ax):
    """Draw the Accra basemap into ax and strip the plot furniture."""
    ax.imshow(basemap, extent=EXTENT, zorder=0)
    ax.set_xlim(EXTENT[0], EXTENT[1])
    ax.set_ylim(EXTENT[2], EXTENT[3])
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)


def coords_of(names):
    """Longitude and latitude arrays for a list of site names, in that order."""
    chosen = sites.loc[list(names)]
    return chosen["longitude"].values, chosen["latitude"].values

### 3. The station map: colour carries the number

The first real map. Each sensor becomes a dot, and the **colour** of the dot is
its mean concentration for the month — what people usually mean by a "heatmap"
of a sensor network.

Two design rules do most of the work here:

* **One hue, light to dark.** Concentration is a magnitude — more is more — so
  it gets a single-hue ramp that runs pale to deep. A rainbow would invent
  categories that the data doesn't have.
* **Both months share one colour scale.** If February ran on its own scale and
  August on its own, the two maps would look identical and the comparison would
  be a lie. One scale for both, always.

In [ ]:
# A red ramp that stops short of paper-white, so the palest sensor is still
# visible against a light grey basemap.
HEAT = LinearSegmentedColormap.from_list("heat", plt.cm.Reds(np.linspace(0.30, 0.97, 256)))

feb_means = feb.groupby("site_name")["pm25_value"].mean()
aug_means = aug.groupby("site_name")["pm25_value"].mean()

VMIN = np.floor(min(feb_means.min(), aug_means.min()))
VMAX = np.ceil(max(feb_means.max(), aug_means.max()))

print(f"Shared colour scale: {VMIN:.0f} to {VMAX:.0f} µg/m³")
print(f"February mean across sites: {feb_means.mean():.1f} µg/m³")
print(f"August mean across sites:   {aug_means.mean():.1f} µg/m³\n")
print("February 2025, dirtiest first:")
print(feb_means.sort_values(ascending=False).round(1).to_string())

Now the map. One extra touch: sensor names, placed so they don't sit on top of
each other. Automatic label placement is a genuinely fiddly problem, so the
helper below is given to you — it scores eight positions around each dot and
keeps the least-crowded one.

In [ ]:
def label_sites(ax, lon, lat, names, fontsize=7.2, marker_pad_pt=9.0):
    """Draw site labels, each in the least-crowded slot available to it.

    Eight candidate positions are scored per point (overlap with the markers and
    with labels already placed, plus a penalty for falling outside the panel) and
    the best-scoring one wins. Boxes are measured in points and converted to data
    units through the axes' own size, so the estimate holds at any figure size.
    """
    ax.figure.canvas.draw()  # a laid-out axes is needed to measure against
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    box_px = ax.get_window_extent()
    x_per_pt = (x1 - x0) / box_px.width * ax.figure.dpi / 72.0
    y_per_pt = (y1 - y0) / box_px.height * ax.figure.dpi / 72.0

    char_w = 0.52 * fontsize * x_per_pt
    line_h = 1.30 * fontsize * y_per_pt
    pad_x, pad_y = marker_pad_pt * x_per_pt, marker_pad_pt * y_per_pt
    # Eight directions at two distances. The far ring is what saves a dense
    # cluster: with one ring only, crowded labels have nowhere to go and end up
    # stacked on their neighbours.
    dirs = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (-1, 1), (1, -1), (-1, -1)]
    slots = [(dx, dy, 1.0) for dx, dy in dirs] + [(dx, dy, 2.1) for dx, dy in dirs]
    taken = [(x - pad_x, y - pad_y, x + pad_x, y + pad_y) for x, y in zip(lon, lat)]

    def overlap(a, b):
        return (max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
                * max(0.0, min(a[3], b[3]) - max(a[1], b[1])))

    # Longest names first: they are the hardest to place, so let them claim a
    # clear slot before the short ones fill the space up.
    order = sorted(range(len(names)), key=lambda i: -len(str(list(names)[i])))
    lon, lat, names = list(lon), list(lat), list(names)
    for i in order:
        x, y, name = lon[i], lat[i], names[i]
        w, h = char_w * len(str(name)), line_h
        best, best_score = None, None
        for fx, fy, ring in slots:
            cx = x + fx * (w / 2 + pad_x) * ring
            cy = y + fy * (h / 2 + pad_y) * ring
            box = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
            score = sum(overlap(box, t) for t in taken) / (char_w * line_h)
            outside = ((max(0.0, x0 - box[0]) + max(0.0, box[2] - x1)) / x_per_pt
                       + (max(0.0, y0 - box[1]) + max(0.0, box[3] - y1)) / y_per_pt)
            score += 50.0 * outside + 0.6 * (ring - 1.0)  # prefer the near ring
            if best_score is None or score < best_score:
                best, best_score = (cx, cy, box), score
        cx, cy, box = best
        taken.append(box)
        ax.plot([x, cx], [y, cy], color="#999999", linewidth=0.6, zorder=3)
        ax.annotate(str(name), (cx, cy), ha="center", va="center", fontsize=fontsize,
                    color="#333333", zorder=5,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.85))


def station_map(ax, means, title):
    """Sensors as dots, coloured by mean concentration."""
    lon, lat = coords_of(means.index)
    dots = ax.scatter(lon, lat, c=means.values, cmap=HEAT, vmin=VMIN, vmax=VMAX,
                      s=170, edgecolor="white", linewidth=1.8, zorder=4)
    label_sites(ax, lon, lat, means.index, fontsize=6.8)
    ax.set_title(title)
    return dots


fig, (ax_feb, ax_aug) = plt.subplots(1, 2, figsize=(13.5, 7.0))
map_panel(ax_feb)
map_panel(ax_aug)
station_map(ax_feb, feb_means, f"February 2025   ({len(feb_means)} sites, mean {feb_means.mean():.0f})")
dots = station_map(ax_aug, aug_means, f"August 2025   ({len(aug_means)} sites, mean {aug_means.mean():.0f})")
bar = fig.colorbar(dots, ax=[ax_feb, ax_aug], fraction=0.033, pad=0.02)
bar.set_label("Mean PM2.5 (µg/m³)")
bar.outline.set_visible(False)
fig.suptitle("August is far cleaner across the city — with one stubborn exception",
             fontsize=13, fontweight="bold", x=0.015, ha="left")
plt.show()

The city-wide mean falls hard between the seasons, from about 32 to about 22
µg/m³ — the Harmattan lifts, the rains arrive, and most of Accra gets cleaner.

Then look at **Agbogbloshie**, the scrap-metal market and industrial district
just west of the centre. It is 45.4 µg/m³ in February and **46.9 in August** — it
does not care what season it is.

Set that against every other site measured in both months: they fall by between
16% and 41%, a median drop of **31%**. Agbogbloshie *rises* 3%, and in doing so
goes from being the second-dirtiest site in the network to comfortably the
dirtiest.

That single stubborn dot is the whole notebook in miniature. Whatever dominates
Agbogbloshie is not the thing that leaves when the rains come.

### ✏️ Your turn 1: the spread within each month

A map shows you the pattern; a number tells you how big it is. For each month,
compute the **range** of site means — the dirtiest site minus the cleanest.

Store February's in `feb_spread` and August's in `aug_spread`.

In [ ]:
# the site means are already computed, above: feb_means and aug_means
feb_spread = ...
aug_spread = ...

print("February spread:", feb_spread)
print("August spread:  ", aug_spread)

In [ ]:
check("feb_spread", lambda: abs(feb_spread - 22.6) < 1.0,
      "dirtiest minus cleanest — feb_means has .max() and .min() methods")
check("aug_spread", lambda: abs(aug_spread - 32.5) < 1.0,
      "the same calculation, on aug_means")

Read those two numbers again, because they are the opposite of what most people
expect. August is the **cleaner** month, and it is also the **more unequal** one
— the gap between Accra's best and worst sites is about 33 µg/m³ in August
against about 23 in February.

It makes sense once you see why. In February a regional dust load sits over
every site at once, raising them all together and compressing the differences.
When that shared load lifts, what's left is whatever each neighbourhood makes
for itself — and *that* varies enormously. Notebook 8 measures both halves.

### 4. Filling the gaps: inverse-distance weighting

Seventeen dots leave a lot of city uncovered. What would we estimate *between*
them?

The simplest honest answer is **inverse-distance weighting (IDW)**: the estimate
at any spot is an average of the sensors, where nearer sensors count for more.
One line of maths:

$$\hat{z}(x) = \frac{\sum_i w_i z_i}{\sum_i w_i} \qquad w_i = \frac{1}{d_i^{\,p}}$$

where $d_i$ is the distance to sensor $i$ and $p$ (the *power*) sets how fast
influence falls off. $p = 2$ is the usual starting point: a sensor twice as far
away gets a quarter of the say.

Two practical wrinkles:

* **Degrees aren't kilometres.** One degree of latitude is ~110.6 km anywhere,
  but one degree of longitude shrinks as you leave the equator. Accra sits at
  5.6°N, so a degree of longitude is still ~110.7 km — nearly the same, but the
  conversion is worth doing properly.
* **Never divide by zero.** Right on top of a sensor, $d = 0$. Clamping the
  distance to a tiny floor keeps the arithmetic finite.

In [ ]:
def idw(grid_lon, grid_lat, lon, lat, values, power=2):
    """Inverse-distance-weighted estimate at every point of a grid."""
    km_per_lon = 111.32 * np.cos(np.deg2rad(lat.mean()))
    dx = (grid_lon[..., None] - lon) * km_per_lon
    dy = (grid_lat[..., None] - lat) * 110.57
    dist = np.sqrt(dx ** 2 + dy ** 2)
    weights = 1.0 / np.maximum(dist, 1e-6) ** power
    return (weights * values).sum(axis=-1) / weights.sum(axis=-1)


# A quick sanity check before trusting it on a whole grid: asked for the value
# *at* a sensor, IDW should return that sensor's own value.
lon_f, lat_f = coords_of(feb_means.index)
at_first = idw(np.array([lon_f[0]]), np.array([lat_f[0]]), lon_f, lat_f, feb_means.values)
print(f"IDW at {feb_means.index[0]}: {at_first[0]:.2f} µg/m³")
print(f"measured there:              {feb_means.values[0]:.2f} µg/m³")

Good — the interpolator honours the data it was given.

Now, where should we *draw* it? IDW will happily produce a number 50 km out to
sea, built from sensors 50 km away. That number is arithmetic, not evidence. The
standard guard is to draw only inside the **convex hull** of the sensors — the
rubber-band outline stretched around the network — and leave the rest blank.

In [ ]:
from matplotlib.path import Path as MplPath
from scipy.spatial import ConvexHull


def hull_mask(grid_lon, grid_lat, lon, lat, grow=1.10):
    """True inside the sensors' convex hull, expanded slightly about its centre."""
    points = np.column_stack([lon, lat])
    outline = points[ConvexHull(points).vertices]
    centre = outline.mean(axis=0)
    outline = centre + (outline - centre) * grow
    inside = MplPath(outline).contains_points(
        np.column_stack([grid_lon.ravel(), grid_lat.ravel()]))
    return inside.reshape(grid_lon.shape), outline


grid_lon, grid_lat = np.meshgrid(np.linspace(EXTENT[0], EXTENT[1], 360),
                                 np.linspace(EXTENT[2], EXTENT[3], 360))


def idw_map(ax, means, title, power=2):
    lon, lat = coords_of(means.index)
    surface = idw(grid_lon, grid_lat, lon, lat, means.values, power=power)
    inside, outline = hull_mask(grid_lon, grid_lat, lon, lat)
    image = ax.imshow(np.where(inside, surface, np.nan), extent=EXTENT, origin="lower",
                      cmap=HEAT, vmin=VMIN, vmax=VMAX, alpha=0.8, zorder=2)
    ax.plot(np.append(outline[:, 0], outline[0, 0]),
            np.append(outline[:, 1], outline[0, 1]),
            color="#555555", linewidth=0.9, linestyle="--", zorder=3)
    ax.scatter(lon, lat, facecolor="none", edgecolor="#222222",
               linewidth=1.3, s=55, zorder=4)
    ax.set_title(title)
    return image


fig, (ax_feb, ax_aug) = plt.subplots(1, 2, figsize=(13.5, 7.0))
map_panel(ax_feb)
map_panel(ax_aug)
idw_map(ax_feb, feb_means, f"February 2025   ({len(feb_means)} sites)")
image = idw_map(ax_aug, aug_means, f"August 2025   ({len(aug_means)} sites)")
bar = fig.colorbar(image, ax=[ax_feb, ax_aug], fraction=0.033, pad=0.02)
bar.set_label("Interpolated mean PM2.5 (µg/m³)")
bar.outline.set_visible(False)
fig.suptitle("Inverse-distance weighting, drawn only where sensors surround the estimate",
             fontsize=13, fontweight="bold", x=0.015, ha="left")
plt.show()

February is a broad, fairly even wash of red — consistent with one big source
affecting everywhere. August is mostly pale with a hard **bullseye** over
Agbogbloshie.

Be careful with that bullseye, though. It is partly real (Agbogbloshie genuinely
is that much dirtier) and partly an artefact of the method: with $p = 2$ every
sensor pulls the surface hard towards its own value nearby, so every station
becomes a little peak or pit. IDW draws a smooth surface; it knows no
atmospheric physics, and it will never invent a hotspot *between* two sensors.

Treat these surfaces as an interpolation of your measurements, not as a
concentration field. The dashed hull is the edge of what you actually know.

### ✏️ Your turn 2: what does the power do?

Redraw August's surface with `power=1` (influence falls off slowly) and with
`power=5` (each sensor dominates its immediate area). Put them side by side.

Use `idw_map(ax, aug_means, title, power=...)` — it takes the power directly.

In [ ]:
fig, (ax_p1, ax_p5) = plt.subplots(1, 2, figsize=(13.5, 7.0))
map_panel(ax_p1)
map_panel(ax_p5)
# call idw_map twice, once into ax_p1 with power=1, once into ax_p5 with power=5
soft = ...
sharp = ...
plt.show()

With `power=1` the map drifts towards the city-wide average and Agbogbloshie's
hotspot smears across half the city. With `power=5` it breaks into flat tiles
with abrupt seams — essentially "nearest sensor wins". Neither extreme is wrong,
but both are *choices*, and a reader cannot see your choice from the picture.
Say which power you used whenever you publish one of these.

### 5. Do neighbouring sensors agree?

The maps so far average a whole month into one number per site. That throws away
the thing we most want to know: do two nearby sensors go up and down **at the
same times**?

That's a correlation. For each sensor we'll find its nearest neighbour, line the
two hourly series up, and compute Pearson's *r*:

* *r* near **1** — the two sites rise and fall together. Something city-wide is
  driving both.
* *r* near **0** — they move independently. Each is responding to its own local
  sources… *or* one of them is broken. Hold that thought.

First, distances on a sphere. The haversine formula is the standard tool:

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometres."""
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dphi = p2 - p1
    dlam = np.deg2rad(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def nearest_neighbour_pairs(df):
    """For every site: its closest other site, the distance, and their correlation."""
    wide = df.pivot_table(index="datetime", columns="site_name", values="pm25_value")
    names = list(wide.columns)
    rows = []
    for site in names:
        others = [other for other in names if other != site]
        distances = [haversine_km(sites.loc[site, "latitude"], sites.loc[site, "longitude"],
                                  sites.loc[other, "latitude"], sites.loc[other, "longitude"])
                     for other in others]
        nearest = others[int(np.argmin(distances))]
        rows.append(dict(site=site, neighbour=nearest, km=min(distances),
                         r=wide[site].corr(wide[nearest]),
                         shared_hours=int(wide[[site, nearest]].dropna().shape[0])))
    return pd.DataFrame(rows).sort_values("km").reset_index(drop=True)


feb_pairs = nearest_neighbour_pairs(feb)
aug_pairs = nearest_neighbour_pairs(aug)

print("FEBRUARY 2025")
print(feb_pairs.round(2).to_string(index=False))
print(f"\nmedian r = {feb_pairs['r'].median():.2f}")

In [ ]:
print("AUGUST 2025")
print(aug_pairs.round(2).to_string(index=False))
print(f"\nmedian r = {aug_pairs['r'].median():.2f}")

Median *r* of about **0.63** in February and **0.59** in August — barely
different. Accra's sensors share a rhythm all year round, even though the
*levels* they sit at diverge sharply in August.

That's an important pairing to hold in your head, because the two facts sound
contradictory and aren't:

* **the same rhythm** — when a wet-season squall clears the air, it clears it
  over the whole city at once, so every sensor dips at the same hour;
* **different levels** — but in August Agbogbloshie averages 46.9 µg/m³ while
  West Hills Mall averages 14.4, a factor of three apart all month.

Correlation measures the rhythm. It says nothing at all about the level. Two
sites can correlate at 0.9 and still differ threefold — which is exactly what
these two do.

Now put it on the map, drawing each sensor's link to its nearest neighbour and
colouring the link by *r*.

In [ ]:
COOL = LinearSegmentedColormap.from_list("cool", plt.cm.Blues(np.linspace(0.28, 0.97, 256)))
R_NORM = plt.Normalize(0, 1)


def correlation_map(ax, pairs, title):
    for _, row in pairs.iterrows():
        lons = [sites.loc[row.site, "longitude"], sites.loc[row.neighbour, "longitude"]]
        lats = [sites.loc[row.site, "latitude"], sites.loc[row.neighbour, "latitude"]]
        ax.plot(lons, lats, color=COOL(R_NORM(row.r)), linewidth=4.0,
                solid_capstyle="round", zorder=2)
    lon, lat = coords_of(pairs["site"])
    ax.scatter(lon, lat, s=60, color="#333333", zorder=4)
    label_sites(ax, lon, lat,
                [f"{s}  r={r:.2f}" for s, r in zip(pairs["site"], pairs["r"])],
                fontsize=6.4)
    ax.set_title(title)


fig, (ax_feb, ax_aug) = plt.subplots(1, 2, figsize=(13.5, 7.0))
map_panel(ax_feb)
map_panel(ax_aug)
correlation_map(ax_feb, feb_pairs, f"February 2025   (median r = {feb_pairs['r'].median():.2f})")
correlation_map(ax_aug, aug_pairs, f"August 2025   (median r = {aug_pairs['r'].median():.2f})")
bar = fig.colorbar(plt.cm.ScalarMappable(cmap=COOL, norm=R_NORM),
                   ax=[ax_feb, ax_aug], fraction=0.033, pad=0.02)
bar.set_label("Hourly correlation with nearest neighbour (r)")
bar.outline.set_visible(False)
fig.suptitle("Accra's sensors keep step with each other in both seasons — almost all of them",
             fontsize=13, fontweight="bold", x=0.015, ha="left")
plt.show()

Almost every link is a confident blue in both panels. But scan the August map
for the one that isn't — a link so pale it is nearly white, at **Osu Presby
School**, *r* = 0.01. Its nearest neighbour is 1.1 km away and they agree on
essentially nothing.

Two explanations fit. Either Osu Presby School has an extraordinary local source
that no other site sees… or the sensor is broken. The next section tells you how
to find out, and the answer matters: one is a finding, the other is a fault.

### 6. Is it the air, or is it the instrument?

Before believing any spatial result, a professional asks the awkward question:
could this pattern come from the instruments rather than the atmosphere?

Two cheap tests answer it, and this network hands us both.

**Test 1 — colocation.** Two PurpleAir units at the Afri-SET sensor-evaluation
facility sit **5 metres** apart. They breathe identical air. Anything they
disagree about is instrument noise, full stop.

In [ ]:
def grid_hourly(df, start, end):
    """Pivot to one column per site on a gap-free hourly index."""
    wide = df.pivot_table(index="datetime", columns="site_name", values="pm25_value")
    return wide.reindex(pd.date_range(start, end, freq="h", tz="UTC"))


feb_wide = grid_hourly(feb, "2025-02-01", "2025-02-28 23:00")
aug_wide = grid_hourly(aug, "2025-08-01", "2025-08-31 23:00")

for label, wide in [("February", feb_wide), ("August", aug_wide)]:
    pair = wide[["Afri-SET CC1", "Afri-SET F1"]].dropna()
    hourly_r = pair.corr().iloc[0, 1]
    daily_r = pair.resample("D").mean().corr().iloc[0, 1]
    print(f"{label}: {len(pair)} shared hours   hourly r = {hourly_r:.2f}   "
          f"daily r = {daily_r:.2f}   means {pair.mean().round(1).tolist()}")

**r = 0.99 and 1.00.** That is what two healthy sensors in the same air look
like, and it is the yardstick everything else gets measured against. Whatever is
happening at Osu Presby School, we now know the network's instruments are
*capable* of near-perfect agreement.

(This matters more than it may seem. Run the same colocation test on some other
networks and you get 0.3, not 0.99 — at which point every spatial conclusion you
were about to draw is worth much less.)

**Test 2 — autocorrelation.** Real ambient PM2.5 is smooth: this hour resembles
last hour, because air masses take time to move. A series that jumps around
randomly hour to hour is measuring something other than the atmosphere.

In [ ]:
print("Correlation between each hour and the hour before it")
print(f"{'site':<28}{'February':>10}{'August':>9}")
for site in sorted(set(feb_wide.columns) | set(aug_wide.columns)):
    f = feb_wide[site].autocorr(1) if site in feb_wide else float("nan")
    a = aug_wide[site].autocorr(1) if site in aug_wide else float("nan")
    print(f"{site:<28}{f:>10.2f}{a:>9.2f}")
print(f"\n{'MEDIAN':<28}"
      f"{np.nanmedian([feb_wide[c].autocorr(1) for c in feb_wide.columns]):>10.2f}"
      f"{np.nanmedian([aug_wide[c].autocorr(1) for c in aug_wide.columns]):>9.2f}")

The network as a whole is healthy: a median of **0.79** in February and **0.77**
in August, and every site sits in a tight band from about 0.59 to 0.93.

Every site but one. **Osu Presby School: 0.06.** Its reading this hour tells you
essentially nothing about its reading next hour. Ambient air does not behave
that way; a faulty sensor does.

Notice what makes this fault nasty. It passed the completeness screen (73% of
August's hours). Its mean, 24.3 µg/m³, is unremarkable. Its range, 3.4 to 85.2,
looks perfectly normal. **Nothing about the distribution gives it away** — only
the *time ordering* does. A range check or a completeness check would have waved
it straight through.

### ✏️ Your turn 3: find the broken sensor from the data

Suppose nobody had told you. Compute the lag-1 autocorrelation of every site in
`aug_wide` and find the **lowest** one.

Store the site's name in `suspect`.

In [ ]:
# build a Series of autocorrelations, one per column, then find the smallest.
# a dict comprehension over aug_wide.columns is one readable way in.
autocorrs = ...
suspect = ...

print("Most suspicious sensor:", suspect)

In [ ]:
check("suspect", lambda: suspect == "Osu Presby School",
      "aug_wide[site].autocorr(1) per column, then find the label of the minimum")

What should you do with it? For this notebook, nothing — it has been left in on
purpose so you could find it. In a real analysis you would drop it, and say so
in your methods.

One reassurance about the maps you have already drawn: this sensor is one of
twenty, and the interpolated surface barely notices it. But had you been
comparing two sites, and had one of them been this one, you would have published
a fault as a finding.

### 7. What you built

Three maps, each answering a different question:

| Map | Question | February | August |
|---|---|---|---|
| Station means | Where is it worst? | mean ~32, spread ~23 | mean ~22, spread ~33 |
| IDW surface | What's between the sensors? | broad even wash | pale, with an Agbogbloshie bullseye |
| Nearest-neighbour *r* | Do neighbours agree? | median 0.63 | median 0.59 |

And three habits worth more than the maps:

1. **One colour scale across everything you're comparing.** Otherwise the
   picture flatters whichever panel you scaled last.
2. **Draw interpolation only where you have evidence.** The convex hull is the
   boundary of what your network can honestly claim.
3. **Test the instruments before you trust the pattern.** A colocated pair and a
   column of autocorrelations cost you five lines, and here they caught a sensor
   that every other check had passed.

The finding to carry forward is the one from section 3: August is cleaner *and*
more unequal. Something big and shared disappears between February and August,
and what it leaves behind is a city of very different neighbourhoods. In
**Notebook 8** you'll separate those two things — the shared regional background
and each site's own local contribution — and measure them both.